# 🎬 AI Video Motion Control — Google Colab
**Run the full Character Replacement pipeline on a free T4 GPU.**

Steps:
1. Connect a **GPU runtime** (Runtime → Change runtime type → T4 GPU)
2. Run all cells in order
3. Use the public Gradio link to access the UI from any device

In [ ]:
#@title 1. Install System Dependencies
!apt-get -y install -qq aria2
!nvidia-smi

In [ ]:
#@title 2. Clone Project + ComfyUI
import os
os.chdir('/content')

# Clone our project
!git clone https://github.com/sametkzl22/motion-control.git
os.chdir('/content/motion-control')

# Clone ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git

# Install ComfyUI deps
!pip install -q -r ComfyUI/requirements.txt
!pip install -q -r requirements.txt
!pip install insightface -q

# Install PyTorch with CUDA
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
#@title 3. Install Custom Nodes
CUSTOM = '/content/motion-control/ComfyUI/custom_nodes'

!git clone https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git {CUSTOM}/ComfyUI-AnimateDiff-Evolved
!git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git {CUSTOM}/comfyui_controlnet_aux
!pip install -q -r {CUSTOM}/comfyui_controlnet_aux/requirements.txt 2>/dev/null || true
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git {CUSTOM}/ComfyUI-VideoHelperSuite
!pip install -q -r {CUSTOM}/ComfyUI-VideoHelperSuite/requirements.txt 2>/dev/null || true
!git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git {CUSTOM}/ComfyUI_IPAdapter_plus

In [ ]:
#@title 4. Download Models (uses aria2 for speed)
MODELS = '/content/motion-control/ComfyUI/models'

# SD 1.5 Checkpoint (Light Pruned)
!aria2c -x 16 -s 16 -d {MODELS}/checkpoints -o v1-5-pruned.safetensors \
  https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned.safetensors

# AnimateDiff
!aria2c -x 16 -s 16 -d {MODELS}/animatediff_models -o mm_sd_v15_v2.ckpt \
  https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt

# ControlNet OpenPose, Depth, SoftEdge
!aria2c -x 16 -s 16 -d {MODELS}/controlnet -o control_v11p_sd15_openpose.pth \
  https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_openpose.pth
!aria2c -x 16 -s 16 -d {MODELS}/controlnet -o control_v11f1p_sd15_depth.pth \
  https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth
!aria2c -x 16 -s 16 -d {MODELS}/controlnet -o control_v11p_sd15_softedge.pth \
  https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_softedge.pth

# IP-Adapter Face Plus + CLIP Vision
!mkdir -p {MODELS}/ipadapter {MODELS}/clip_vision
!aria2c -x 16 -s 16 -d {MODELS}/ipadapter -o ip-adapter-plus-face_sd15.safetensors \
  https://huggingface.co/h94/IP-Adapter/resolve/main/models/ip-adapter-plus-face_sd15.safetensors
!aria2c -x 16 -s 16 -d {MODELS}/clip_vision -o sd1.5_model.safetensors \
  https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors

# LCM-LoRA
!mkdir -p {MODELS}/loras
!aria2c -x 16 -s 16 -d {MODELS}/loras -o lcm-lora-sdv1-5.safetensors \
  https://huggingface.co/latent-consistency/lcm-lora-sdv1-5/resolve/main/pytorch_lora_weights.safetensors

print('✅ All models downloaded!')

In [ ]:
#@title 5. Patch config.py for Colab (GPU mode, higher res)
import os
os.chdir('/content/motion-control')

config_patch = '''
# ── Colab GPU Overrides ──────────────────────────────
DEVICE = "cuda"
DEFAULT_WIDTH = 512
DEFAULT_HEIGHT = 512
DEFAULT_FRAMES = 16
MAX_FRAMES = 24
'''

with open('config.py', 'a') as f:
    f.write(config_patch)

print('✅ Config patched for Colab GPU')

In [ ]:
#@title 6. Start ComfyUI Backend (background)
import subprocess, time

comfyui_proc = subprocess.Popen(
    ['python', 'ComfyUI/main.py',
     '--listen', '127.0.0.1',
     '--port', '8188',
     '--force-fp16',
     '--preview-method', 'auto'],
    cwd='/content/motion-control',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for ComfyUI to be ready
import requests
for i in range(60):
    try:
        r = requests.get('http://127.0.0.1:8188/system_stats', timeout=2)
        if r.status_code == 200:
            print(f'✅ ComfyUI ready in {i*2}s')
            break
    except:
        pass
    time.sleep(2)
else:
    print('❌ ComfyUI failed to start')

In [ ]:
#@title 7. Launch Gradio UI (with public link)
import os
os.chdir('/content/motion-control')

# Monkey-patch launch to use share=True
import app as motion_app
gradio_app = motion_app.create_app()
gradio_app.launch(
    server_name='0.0.0.0',
    server_port=7860,
    share=True,
    show_error=True,
    css=motion_app.CSS,
)